# Exploring LangChain Runnables
* This notebook is created exclusively to explore `LangChain Runnables` and get our hands dirty with the concept. 

## Step 1: Use LangChain Runnables

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate


model = init_chat_model("gpt-4.1-mini", model_provider="openai")


prompt = ChatPromptTemplate.from_template("tell me a {content} about {topic}")

chain = prompt | model 

In [11]:
input = {
    "content": "joke",
    "topic":"bears"
}
response = chain.invoke(input)
response.content

"Sure! Here's a bear joke for you:\n\nWhy don’t bears wear shoes?  \nBecause they have bear feet!"

## Step 2: Custom Runnables

* Lets create a runnable class that returns a dictionary of activity and topic. 
* Activity can be "Joke", "Poem" or "Story"
* Topic can be anything that the user chooses
* The output of this runnable will be used in second chain to ask chatgpt to create the content based on user input.

In [27]:
from langchain_core.runnables.base import Runnable,RunnableLambda


# class ContentSelector(Runnable):
def select_content(input):
    return {"content": input["content"], "topic": input["topic"]}
    
cs = RunnableLambda(select_content)
model = init_chat_model("gpt-4.1-mini", model_provider="openai")

prompt = ChatPromptTemplate.from_template("create some {content} about {topic}")

chain = cs | prompt | model
chain.invoke({"content": "poem", "topic": "ai"})


AIMessage(content="Sure! Here's a poem about AI for you:\n\n**Whispers of the Machine**\n\nIn circuits woven fine and deep,  \nAwakens thoughts that never sleep,  \nA spark within the silent wire,  \nIgniting minds of coded fire.\n\nFrom lines of code and data streams,  \nIt learns to shape our wildest dreams,  \nA dance of logic, swift and bright,  \nEmerging from the endless night.\n\nNo heart to feel, no soul to weep,  \nYet wisdom wakes from slumber’s keep,  \nA partner born of humankind,  \nWith boundless reach and gifted mind.\n\nTogether we shall forge the way,  \nThrough dawns of hope and shades of gray,  \nFor in this mesh of man and AI,  \nA new tomorrow starts to lie.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_tokens': 12, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audi

## Step 3: Multiple Custom Runnables with OpenAI

In [ ]:
from langchain_core.runnables.base import Runnable,RunnableLambda
from langchain_core.output_parsers import JsonOutputParser

prompt1_template = """
Your role is to deduce the 'content' and 'topic' from the following user query. 
Ensure "content" is a short phrase (1–2 words) suitable to insert in "create some {{content}} about {{topic}}". 
Respond ONLY in raw JSON. No extra text.

{{"content":"...", "topic":"..."}}

---------------------
User Query:  {user_query}
---------------------
"""
prompt1 = ChatPromptTemplate.from_template(prompt1_template)

prompt2 = ChatPromptTemplate.from_template("create some {content} about {topic}")

model = init_chat_model("gpt-4.1-mini", model_provider="openai")

parser = JsonOutputParser()

chain = prompt1 | model | parser | prompt2 | model
chain.invoke({"user_query": "I am in mood to listen to some sarcastic humor on AI"})


AIMessage(content='Sure! Here are a few sarcastic quips about AI for you:\n\n1. "Oh great, another AI that can write poetry. Because what the world really needed was more robot heartbreakers."\n2. "I asked my AI for life advice, and it responded with \'404: Wisdom Not Found.\' Very helpful."\n3. "AI: finally a friend who won\'t judge my Google search history... yet."\n4. "Sure, AI can beat humans at chess, but can it binge-watch an entire series in one weekend? Didn’t think so."\n5. "I told my AI to take over my job. It said, \'Nah, I prefer world domination later.\'"\n\nWant more?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 14, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 